In [ ]:
# Install PySpark
!pip install pyspark

# Import libraries
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
    .appName("Sprint5Project") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# Check version
print("Spark Version:", spark.version)

Spark Version: 4.0.2


In [ ]:
from google.colab import files
files.upload()

In [ ]:
df = spark.read.csv(
    "amz_ca_total_products_data_processed.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/amz_ca_total_products_data_processed.csv. SQLSTATE: 42K03

In [ ]:
df.printSchema()
print("Row count:", df.count())
print("Column count:", len(df.columns))
df.show(5, truncate=False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType


print("Initial rows:", df.count())


null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
])
null_counts.show(truncate=False)


string_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType()"]
for c in string_cols:
    df = df.withColumn(c, F.trim(F.col(c)))


numeric_casts = {
    "price": DoubleType(),
    "listPrice": DoubleType(),
    "stars": DoubleType(),
    "reviews": IntegerType(),
    "boughtInLastMonth": IntegerType(),
}
for col_name, dtype in numeric_casts.items():
    if col_name in df.columns:
        df = df.withColumn(col_name, F.col(col_name).cast(dtype))


before = df.count()
df = df.dropDuplicates()
print(f"Dropped {before - df.count()} duplicate rows")


if "price" in df.columns:
    df = df.filter((F.col("price") >= 0) | F.col("price").isNull())
if "stars" in df.columns:
    df = df.filter(((F.col("stars") >= 0) & (F.col("stars") <= 5)) | F.col("stars").isNull())
if "reviews" in df.columns:
    df = df.filter((F.col("reviews") >= 0) | F.col("reviews").isNull())


fill_map = {}
if "reviews" in df.columns: fill_map["reviews"] = 0
if "boughtInLastMonth" in df.columns: fill_map["boughtInLastMonth"] = 0
if "isBestSeller" in df.columns: fill_map["isBestSeller"] = False
if fill_map:
    df = df.fillna(fill_map)


df.cache()
print("Cleaned rows:", df.count())
df.printSchema()

In [ ]:
category_col = "category_id" if "category_id" in df.columns else "category"

category_summary = (
    df.groupBy(category_col)
      .agg(
          F.count("*").alias("product_count"),
          F.round(F.avg("price"), 2).alias("avg_price"),
          F.round(F.avg("stars"), 2).alias("avg_rating"),
          F.sum("reviews").alias("total_reviews"),
          F.sum("boughtInLastMonth").alias("total_bought_last_month") if "boughtInLastMonth" in df.columns else F.lit(None).alias("total_bought_last_month"),
          F.sum(F.col("isBestSeller").cast("int")).alias("bestseller_count") if "isBestSeller" in df.columns else F.lit(None).alias("bestseller_count"),
      )
      .orderBy(F.desc("product_count"))
)

category_summary.show(20, truncate=False)

In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy(category_col).orderBy(F.desc("stars"), F.desc("reviews"))

top_products_per_category = (
    df.filter(F.col("reviews") >= 50)
      .withColumn("rank_in_category", F.row_number().over(w))
      .filter(F.col("rank_in_category") <= 5)
      .select(category_col, "title", "stars", "reviews", "rank_in_category")
)

top_products_per_category.show(20, truncate=False)

products_with_category_stats = (
    df.join(
        category_summary.select(
            category_col,
            F.col("avg_price").alias("category_avg_price"),
            F.col("avg_rating").alias("category_avg_rating"),
        ),
        on=category_col,
        how="left",
    )
    .withColumn("price_vs_category_avg", F.round(F.col("price") - F.col("category_avg_price"), 2))
    .withColumn("rating_vs_category_avg", F.round(F.col("stars") - F.col("category_avg_rating"), 2))
)

products_with_category_stats.select(
    "title", "price", "category_avg_price", "price_vs_category_avg",
    "stars", "category_avg_rating", "rating_vs_category_avg"
).show(10, truncate=False)